In [1]:
import os
os.environ["TMPDIR"] = "/workspace/tmp"
os.environ["HF_HOME"] = "/workspace/hf"
os.environ["HF_DATASETS_CACHE"] = "/workspace/hf/datasets"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.makedirs("/workspace/tmp", exist_ok=True)
os.makedirs("/workspace/hf/datasets", exist_ok=True)


In [ ]:
os.environ["HF_TOKEN"] = "~~"   # 본인 토큰
token = os.getenv("HF_TOKEN")

MODEL_ID = "LGAI-EXAONE/EXAONE-4.0-1.2B"
OUT_DIR  = "/workspace/model_mix_v2"

MAX_SEQ_LEN = 2048


In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, trust_remote_code=True, token=token
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, trust_remote_code=True, torch_dtype=torch.bfloat16, token=token
)
print("model loaded")


/workspace/venv-eval/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!


model loaded


In [4]:
# Cell 0) 설치 (한 번만)
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "peft"])


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 41.1 MB/s  0:00:00


0

In [14]:
# Risk-hardened SFT preprocessing (NO truncation, drop overlength, KEEP ORIGINAL TEXT)
# - prompt/answer 내용 수정 없음
# - 길이 초과 샘플만 제외
# - prompt 구간은 labels=-100, answer 구간만 학습

from collections import Counter

MAX_LEN_TRAIN = 2048
EOS_ID = tokenizer.eos_token_id

def featurize_keep_raw(ex):
    src = ex.get("src", "unknown")
    prompt = (ex.get("prompt", "") or "").strip()
    answer = (ex.get("answer", "") or "").strip()

    if not prompt or not answer:
        return {"keep": False, "src": src, "input_ids": [], "labels": [], "attention_mask": []}

    p_ids = tokenizer(prompt, add_special_tokens=False, truncation=False).input_ids
    a_ids = tokenizer(answer, add_special_tokens=False, truncation=False).input_ids + [EOS_ID]

    total_len = len(p_ids) + len(a_ids)
    if total_len > MAX_LEN_TRAIN:
        # 입력 절단 없이 제외
        return {"keep": False, "src": src, "input_ids": [], "labels": [], "attention_mask": []}

    input_ids = p_ids + a_ids
    labels = [-100] * len(p_ids) + a_ids
    attention_mask = [1] * len(input_ids)

    return {
        "keep": True,
        "src": src,
        "input_ids": input_ids,
        "labels": labels,
        "attention_mask": attention_mask,
    }

assert "sft_raw" in globals(), "sft_raw가 먼저 준비되어 있어야 합니다."

tmp = sft_raw.map(featurize_keep_raw)
before = Counter(tmp["src"])
tmp_keep = tmp.filter(lambda x: x["keep"])
after = Counter(tmp_keep["src"])

print("=== keep ratio by src ===")
for k in sorted(before):
    b = before[k]
    a = after.get(k, 0)
    print(f"{k:12s} {a:4d}/{b:4d} keep={a/b:.1%}")

sft_train = tmp_keep.remove_columns([c for c in tmp_keep.column_names if c in ("keep", "src")])

if len(sft_train) == 0:
    raise RuntimeError("sft_train is empty after filtering.")

max_len_seen = max(len(x) for x in sft_train["input_ids"])
print("final train size:", len(sft_train), "max_len:", max_len_seen)
assert max_len_seen <= MAX_LEN_TRAIN


Filter: 100%|████████████████████████████| 4000/4000 [00:00<00:00, 6223.56 examples/s]


=== keep ratio by src ===
KMMLU-Pro    1000/1000 keep=100.0%
KMMLU-Redux   600/ 600 keep=100.0%
Ko-LongRAG      3/ 400 keep=0.8%
MANTA-1M     2000/2000 keep=100.0%
final train size: 3603 max_len: 1929


In [17]:
# LoRA SFT (shared tensor 저장 에러 방지 버전)
import torch
from transformers import AutoModelForCausalLM, Trainer, TrainingArguments
from peft import LoraConfig, TaskType, get_peft_model

MODEL_ID = "LGAI-EXAONE/EXAONE-4.0-1.2B"
OUT_DIR = "/workspace/lora_answer_sft_2048"

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    token=globals().get("token", None),
)
model.config.use_cache = False
model.gradient_checkpointing_enable()

lora_cfg = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_cfg)

def collate(batch):
    pad_id = tokenizer.pad_token_id
    max_len = max(len(x["input_ids"]) for x in batch)
    input_ids, labels, attention_mask = [], [], []
    for x in batch:
        pad = max_len - len(x["input_ids"])
        input_ids.append(x["input_ids"] + [pad_id] * pad)
        attention_mask.append(x["attention_mask"] + [0] * pad)
        labels.append(x["labels"] + [-100] * pad)
    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
    }

args = TrainingArguments(
    output_dir=OUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=1,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    bf16=True,
    logging_steps=20,
    save_strategy="no",          # 중간 checkpoint 저장 끔
    save_safetensors=False,      # safetensors 저장 비활성화
    report_to="none",
    dataloader_num_workers=0,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=sft_train,
    data_collator=collate,
)

trainer.train()

# 최종 어댑터 저장 (safetensors 강제 비활성)
model.save_pretrained(f"{OUT_DIR}/final_adapter", safe_serialization=False)
tokenizer.save_pretrained(f"{OUT_DIR}/final_adapter")
print("adapter saved:", f"{OUT_DIR}/final_adapter")


/workspace/venv-eval/lib/python3.11/site-packages/torch/utils/data/_utils/pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.pin_memory() is deprecated. Please do not pass this argument. (Triggered internally at /pytorch/aten/src/ATen/native/Memory.cpp:46.)
  return data.pin_memory(device)
/workspace/venv-eval/lib/python3.11/site-packages/torch/utils/data/_utils/pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.is_pinned() is deprecated. Please do not pass this argument. (Triggered internally at /pytorch/aten/src/ATen/native/Memory.cpp:31.)
  return data.pin_memory(device)


Step,Training Loss
20,1.635800
40,0.834100
60,0.772300
80,0.729200
100,0.708000
120,0.643800
140,0.682900
160,0.640200
180,0.659500
200,0.641600


adapter saved: /workspace/lora_answer_sft_2048/final_adapter


In [18]:
# 1) LoRA adapter -> merged 모델 저장
import os, shutil, torch
from peft import PeftModel
from transformers import AutoModelForCausalLM

MODEL_ID = "LGAI-EXAONE/EXAONE-4.0-1.2B"
ADAPTER_DIR = "/workspace/lora_answer_sft_2048/final_adapter"
MERGED_DIR = "/workspace/sft_merged_2048"

if os.path.exists(MERGED_DIR):
    shutil.rmtree(MERGED_DIR)

base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    token=globals().get("token", None),
)
merged = PeftModel.from_pretrained(base, ADAPTER_DIR).merge_and_unload()

# shared tensor 에러 방지
merged.save_pretrained(MERGED_DIR, safe_serialization=False)
tokenizer.save_pretrained(MERGED_DIR)

print("merged saved:", MERGED_DIR)


merged saved: /workspace/sft_merged_2048


In [5]:
import itertools
from collections import Counter
from datasets import load_dataset, Dataset

# ===== 설정 =====
SEED = 42
BUFFER = 10000
MAX_SEQ_LEN = 2048

SAMPLES = {
    "MANTA-1M": 2048,
    "KMMLU-Pro": 1024,     # 권한 없으면 자동 skip
    "KMMLU-Redux": 512,
    "Ko-LongRAG": 512,
}

# ===== 유틸 =====
def pick_split_stream(repo, token=None, splits=("train", "validation", "test")):
    for sp in splits:
        try:
            return load_dataset(repo, split=sp, streaming=True, token=token), sp
        except Exception:
            pass
    ds_dict = load_dataset(repo, streaming=True, token=token)
    first_key = list(ds_dict.keys())[0]
    return ds_dict[first_key], first_key

def sample_stream(ds, n, seed=SEED, buffer=BUFFER):
    ds = ds.shuffle(seed=seed, buffer_size=buffer)
    return list(itertools.islice(ds, n))

def nonempty_text(x):
    return isinstance(x, str) and len(x.strip()) > 0

def format_options(options):
    if isinstance(options, list):
        return "\n".join(f"{chr(65+i)}. {opt}" for i, opt in enumerate(options))
    return str(options)

def pick_answer(options, sol):
    s = str(sol).strip()
    if isinstance(options, list) and len(options) > 0:
        if s.isdigit():
            idx = int(s) - 1
            if 0 <= idx < len(options):
                return str(options[idx])
        if len(s) == 1 and s.upper() in "ABCDE":
            idx = ord(s.upper()) - 65
            if 0 <= idx < len(options):
                return str(options[idx])
    return s

rows = []

# ===== MANTA =====
ds, sp = pick_split_stream("LGAI-EXAONE/MANTA-1M", token=token)
print(f"MANTA-1M split={sp}")
for x in sample_stream(ds, SAMPLES["MANTA-1M"]):
    conv = x.get("conversations", [])
    if not isinstance(conv, list) or len(conv) == 0:   # 빈 배열 배제
        continue
    parts = []
    for turn in conv:
        role = turn.get("role", turn.get("from", "user"))
        content = turn.get("content", turn.get("value", ""))
        if nonempty_text(content):
            parts.append(f"{role}: {content}")
    txt = "\n".join(parts).strip()
    if nonempty_text(txt):
        rows.append({"src": "MANTA-1M", "text": txt})

# ===== KMMLU-Pro =====
try:
    ds, sp = pick_split_stream("LGAI-EXAONE/KMMLU-Pro", token=token)
    print(f"KMMLU-Pro split={sp}")
    for x in sample_stream(ds, SAMPLES["KMMLU-Pro"]):
        q = x.get("question", "")
        opts = x.get("options", [])
        sol = pick_answer(opts, x.get("solution", ""))
        txt = f"{q}\n\n{format_options(opts)}\n\n정답: {sol}".strip()
        if nonempty_text(q) and nonempty_text(sol):
            rows.append({"src": "KMMLU-Pro", "text": txt})
except Exception as e:
    print("KMMLU-Pro skipped:", e)

# ===== KMMLU-Redux =====
ds, sp = pick_split_stream("LGAI-EXAONE/KMMLU-Redux", token=token)
print(f"KMMLU-Redux split={sp}")
for x in sample_stream(ds, SAMPLES["KMMLU-Redux"]):
    q = x.get("question", "")
    opts = x.get("options", [])
    sol = pick_answer(opts, x.get("solution", ""))
    txt = f"{q}\n\n{format_options(opts)}\n\n정답: {sol}".strip()
    if nonempty_text(q) and nonempty_text(sol):
        rows.append({"src": "KMMLU-Redux", "text": txt})

# ===== Ko-LongRAG =====
ds, sp = pick_split_stream("LGAI-EXAONE/Ko-LongRAG", token=token)
print(f"Ko-LongRAG split={sp}")
for x in sample_stream(ds, SAMPLES["Ko-LongRAG"]):
    c = x.get("context", "")
    q = x.get("question", "")
    a = x.get("answer", "")
    txt = f"{c}\n\nQ: {q}\nA: {a}".strip()
    if nonempty_text(txt):
        rows.append({"src": "Ko-LongRAG", "text": txt})

# ===== Dataset 생성 =====
calib_ds = Dataset.from_list(rows).shuffle(seed=SEED)

# 빈 문자열/None 최종 제거
calib_ds = calib_ds.filter(lambda x: nonempty_text(x["text"]))

before = Counter(calib_ds["src"])

# 길이 초과 배제 (절단 없음)
def keep_len(batch):
    enc = tokenizer(
        batch["text"],
        add_special_tokens=True,
        truncation=False,   # 절단 금지
        padding=False,
    )
    return [len(ids) <= MAX_SEQ_LEN for ids in enc["input_ids"]]

calib_ds = calib_ds.filter(
    keep_len,
    batched=True,
    batch_size=64,
    desc=f"Filter <= {MAX_SEQ_LEN} tokens",
)

after = Counter(calib_ds["src"])

print("\n=== Keep ratio by src ===")
for k in sorted(before):
    b = before[k]
    a = after.get(k, 0)
    print(f"{k:12s} {a:4d}/{b:4d} keep={a/b:.1%}")

print("\nfinal calib size:", len(calib_ds))

# 양자화에 바로 쓸 형태(텍스트 컬럼만)
calib_ds_for_quant = calib_ds.remove_columns([c for c in calib_ds.column_names if c != "text"])
print("for quant cols:", calib_ds_for_quant.column_names, "size:", len(calib_ds_for_quant))


MANTA-1M split=train
KMMLU-Pro split=test
KMMLU-Redux split=test
Ko-LongRAG split=test


Filter <= 2048 tokens: 100%|██████████████| 4096/4096 [00:06<00:00, 670.86 examples/s]


=== Keep ratio by src ===
KMMLU-Pro    1024/1024 keep=100.0%
KMMLU-Redux   512/ 512 keep=100.0%
Ko-LongRAG      3/ 512 keep=0.6%
MANTA-1M     2041/2048 keep=99.7%

final calib size: 3580
for quant cols: ['text'] size: 3580


In [7]:
# GPTQ-only 양자화 (NUM_CALIB=256, OOM-safe)
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier
import os

MODEL_IN = "/workspace/sft_merged_2048"          # merge 모델 경로
OUT_DIR = "/workspace/sft_gptq_1024/model"
MAX_SEQ_LEN = 2048                                # 2048에서 터지면 1024 권장
NUM_CALIB = 1024

assert "calib_ds_for_quant" in globals(), "calib_ds_for_quant이 먼저 필요합니다."
assert len(calib_ds_for_quant) >= NUM_CALIB, f"calib size({len(calib_ds_for_quant)}) < 1024"

calib_use = calib_ds_for_quant.shuffle(seed=42).select(range(NUM_CALIB))
os.makedirs(OUT_DIR, exist_ok=True)

recipe = [
    GPTQModifier(
        scheme="W4A16",
        targets=[
            "re:.*q_proj", "re:.*k_proj", "re:.*v_proj", "re:.*o_proj",
            "re:.*gate_proj", "re:.*up_proj", "re:.*down_proj",
        ],
        ignore=["lm_head", "embed_tokens"],
        block_size=64,
        dampening_frac=0.03,
    )
]

oneshot(
    model=MODEL_IN,
    dataset=calib_use,
    recipe=recipe,
    max_seq_length=MAX_SEQ_LEN,
    num_calibration_samples=NUM_CALIB,
    output_dir=OUT_DIR,
)

print("gptq done:", OUT_DIR)


The tokenizer you are loading from '/workspace/sft_merged_2048' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Tokenizing: 100%|█████████████████████████| 1024/1024 [00:01<00:00, 545.60 examples/s]

2026-02-16T06:00:24.072124+0000 | reset | WARNING - Exception during finalizing modifier: Failed to compress 7 modules


2026-02-16T06:00:24.073526+0000 | reset | INFO - Compression lifecycle reset
2026-02-16T06:00:24.075538+0000 | from_modifiers | INFO - Creating recipe from modifiers
2026-02-16T06:00:24.232454+0000 | initialize | INFO - Compression lifecycle initialized for 1 modifiers
2026-02-16T06:00:24.233260+0000 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `GPTQModifier`


(1/31): Calibrating: 100%|███████████████████████| 1024/1024 [00:09<00:00, 102.74it/s]

2026-02-16T06:00:35.917958+0000 | compress_modules | INFO - Quantizing model.layers.0.self_attn.q_proj using 1024 samples


...

2026-02-16T06:11:41.463389+0000 | compress | METRIC - time 0.64s
2026-02-16T06:11:41.466538+0000 | compress | METRIC - error 5517.94
2026-02-16T06:11:41.467662+0000 | compress | METRIC - GPU 0 | usage: 5.52% | total memory: 48 GB
2026-02-16T06:11:41.468178+0000 | compress | METRIC - Compressed module size: 8.486912 MB
2026-02-16T06:11:41.469304+0000 | compress_modules | INFO - Quantizing model.layers.29.self_attn.k_proj using 1024 samples
2026-02-16T06:11:42.099782+0000 | compress | METRIC - time 0.63s
2026-02-16T06:11:42.104700+0000 | compress | METRIC - error 1579.95
2026-02-16T06:11:42.105772+0000 | compress | METRIC - GPU 0 | usage: 5.52% | total memory: 48 GB
2026-02-16T06:11:42.106277+0000 | compress | METRIC - Compressed module size: 2.121728 MB
2026-02-16T06:11:42.107314+0000 | compress_modules | INFO - Quantizing model.layers.29.self_attn.v_proj using 1024 samples
2026-02-16T06:11:42.727711+0000 | compress | METRIC - time 0.62s
2026-02-16T06:11:42.729537+0000 | compress | METR

(31/31): Propagating: 100%|█████████████████████| 1024/1024 [00:00<00:00, 1379.45it/s]

2026-02-16T06:11:53.438134+0000 | finalize | INFO - Compression lifecycle finalized for 1 modifiers
2026-02-16T06:11:53.467317+0000 | get_model_compressor | INFO - skip_sparsity_compression_stats set to True. Skipping sparsity compression statistic calculations. No sparsity compressor will be applied.



Compressing model: 210it [00:06, 31.54it/s]


gptq done: /workspace/sft_gptq_1024/model


In [8]:
# 다음 셀: lm_head 중복 제거 + submit.zip 생성 + 구조 확인
import json, struct, os, shutil, zipfile

MODEL_DIR = "/workspace/sft_gptq_1024/model"
p = f"{MODEL_DIR}/model.safetensors"

# 1) lm_head 중복 제거
with open(p, "rb") as f:
    n = struct.unpack("<Q", f.read(8))[0]
    h = json.loads(f.read(n))

if "lm_head.weight" in h:
    meta = h.get("__metadata__", {})
    keys = [k for k in h if k not in ("__metadata__", "lm_head.weight")]
    new_h = {"__metadata__": meta} if meta else {}
    regions, cur = [], 0

    for k in keys:
        s0, s1 = h[k]["data_offsets"]
        sz = s1 - s0
        new_h[k] = {"dtype": h[k]["dtype"], "shape": h[k]["shape"], "data_offsets": [cur, cur + sz]}
        regions.append((s0, s1))
        cur += sz

    hb = json.dumps(new_h, separators=(",", ":"), ensure_ascii=False).encode("utf-8")
    tmp = p + ".tmp"

    with open(p, "rb") as src, open(tmp, "wb") as dst:
        dst.write(struct.pack("<Q", len(hb)))
        dst.write(hb)
        data_start = 8 + n
        for s0, s1 in regions:
            src.seek(data_start + s0)
            rem = s1 - s0
            while rem:
                c = src.read(min(8 * 1024 * 1024, rem))
                if not c:
                    raise RuntimeError("Unexpected EOF")
                dst.write(c)
                rem -= len(c)
    os.replace(tmp, p)

print("model size GB:", round(os.path.getsize(p) / 1024**3, 3))

# 2) submit.zip 생성
if os.path.exists("/workspace/model"):
    shutil.rmtree("/workspace/model")
shutil.copytree(MODEL_DIR, "/workspace/model")
shutil.make_archive("/workspace/submit", "zip", "/workspace", "model")
print("created:", "/workspace/submit.zip")

# 3) zip 구조 확인
with zipfile.ZipFile("/workspace/submit.zip") as z:
    names = z.namelist()
print("top-level model only:", all(n.startswith("model/") for n in names))
print("safetensors count:", sum(1 for n in names if n.endswith(".safetensors")))
print("has config:", "model/config.json" in names)
print("has tokenizer:", "model/tokenizer.json" in names)


model size GB: 0.905
created: /workspace/submit.zip
top-level model only: True
safetensors count: 1
has config: True
has tokenizer: True


In [9]:
# 스모크 테스트 셀 (HF 로드 + vLLM 생성 + 간단 형식 체크)
from transformers import AutoTokenizer, AutoModelForCausalLM
from vllm import LLM, SamplingParams
import os

MODEL_DIR = "/workspace/model"  # 필요하면 경로 수정

assert os.path.isdir(MODEL_DIR), f"model dir not found: {MODEL_DIR}"

# 1) HF 로드 확인
tok = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True, local_files_only=True)
mdl = AutoModelForCausalLM.from_pretrained(MODEL_DIR, trust_remote_code=True, local_files_only=True)
print("HF load OK")

# 2) vLLM 로드 확인
llm = LLM(model=MODEL_DIR, trust_remote_code=True)
print("vLLM load OK")

# 3) 샘플 질의 (chat template 적용)
prompts = [
    "2+3=? 숫자만 답해.",
    "10-4=? 숫자만 답해.",
    "대한민국 수도는? 한 단어로.",
]

sp = SamplingParams(temperature=0.0, top_p=1.0, max_tokens=24)

for q in prompts:
    msg = [{"role": "user", "content": q}]
    p = tok.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
    out = llm.generate([p], sp)[0].outputs[0].text.strip()
    print(f"\nQ: {q}\nA: {repr(out)}")


The tokenizer you are loading from '/workspace/model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Compressing model: 210it [00:00, 1423.56it/s]


HF load OK
INFO 02-16 06:17:52 [utils.py:263] non-default args: {'trust_remote_code': True, 'disable_log_stats': True, 'model': '/workspace/model'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 02-16 06:17:52 [model.py:530] Resolved architecture: Exaone4ForCausalLM
INFO 02-16 06:17:52 [model.py:1545] Using max model len 65536


2026-02-16 06:17:53,168	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


...


Q: 2+3=? 숫자만 답해.
A: '5'


Adding requests: 100%|█████████████████████████████████| 1/1 [00:00<00:00, 618.36it/s]
Processed prompts: 100%|█| 1/1 [00:00<00:00, 82.17it/s, est. speed input: 2094.60 toks



Q: 10-4=? 숫자만 답해.
A: '10'


Adding requests: 100%|█████████████████████████████████| 1/1 [00:00<00:00, 645.67it/s]
Processed prompts: 100%|█| 1/1 [00:00<00:00, 34.40it/s, est. speed input: 710.12 toks/


Q: 대한민국 수도는? 한 단어로.
A: '대한민국의 수도는 **서울**입니다.'


In [10]:
# (선택) 숫자형/한단어형 아주 간단 pass/fail 체크
import re

def is_short_numeric(s):
    return bool(re.fullmatch(r"\D*([\-]?\d+)\D*", s.strip()))

def is_short_word(s):
    return len(s.strip().split()) <= 3 and len(s.strip()) <= 24

checks = [
    ("2+3=? 숫자만 답해.", is_short_numeric),
    ("10-4=? 숫자만 답해.", is_short_numeric),
    ("대한민국 수도는? 한 단어로.", is_short_word),
]

for q, fn in checks:
    msg = [{"role":"user","content":q}]
    p = tok.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
    a = llm.generate([p], SamplingParams(temperature=0.0, max_tokens=24))[0].outputs[0].text.strip()
    print(q, "->", repr(a), "| PASS" if fn(a) else "| FAIL")


Adding requests: 100%|█████████████████████████████████| 1/1 [00:00<00:00, 835.85it/s]
Processed prompts: 100%|█| 1/1 [00:00<00:00, 101.12it/s, est. speed input: 2433.79 tok


2+3=? 숫자만 답해. -> '5' | PASS


Adding requests: 100%|█████████████████████████████████| 1/1 [00:00<00:00, 638.11it/s]
Processed prompts: 100%|█| 1/1 [00:00<00:00, 85.76it/s, est. speed input: 2129.23 toks


10-4=? 숫자만 답해. -> '10' | PASS


Adding requests: 100%|█████████████████████████████████| 1/1 [00:00<00:00, 696.15it/s]
Processed prompts: 100%|█| 1/1 [00:00<00:00, 33.40it/s, est. speed input: 694.22 toks/

대한민국 수도는? 한 단어로. -> '대한민국의 수도는 **서울**입니다.' | PASS
